# 🌐 XLM-RoBERTa Training for Telugu QA

Train XLM-RoBERTa (XLM-R) on the TeQuAD dataset.

## 📦 Before You Start: What to Upload

Run this command locally to create the training zip:
```bash
python scripts/export_for_colab.py
```

This creates `colab-upload.zip` containing:
```
colab-upload.zip
├── data/
│   └── processed/
│       ├── tequad_train.json      # Training data
│       └── tequad_validation.json # Validation data
└── config/
    └── training_config.yaml       # Training parameters
```

## ⏱️ Training Info

| Model | Parameters | Time | Expected F1 |
|-------|-----------|------|-------------|
| XLM-R | 270M | ~2-3 hours | 77% |

## 📥 After Training: Download These Files

From `models/checkpoints/xlmr/`:
- `config.json`
- `model.safetensors` (or `pytorch_model.bin`)
- `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json`

---

## 1️⃣ Check GPU & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install -q transformers datasets accelerate sentencepiece pyyaml

## 2️⃣ Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create output directory for XLM-R checkpoints
XLMR_OUTPUT = '/content/drive/MyDrive/telugu-qa-XLMR-checkpoints'
os.makedirs(XLMR_OUTPUT, exist_ok=True)
print(f"✓ Checkpoints will be saved to: {XLMR_OUTPUT}")

## 3️⃣ Upload Project Files

In [ ]:
# Upload your colab-upload.zip file
from google.colab import files
import zipfile
import shutil

print("Upload your colab-upload.zip file:")
uploaded = files.upload()

# Extract to a temp folder first
TEMP_DIR = '/content/temp_extract'
os.makedirs(TEMP_DIR, exist_ok=True)

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall(TEMP_DIR)
        print(f"✓ Extracted {filename}")

print("\nExtracted files:")
!ls -la {TEMP_DIR}

In [ ]:
# Fix folder structure - files may have Windows backslash paths
import os
import shutil

TEMP_DIR = '/content/temp_extract'
PROJECT_ROOT = '/content'

# Function to fix Windows-style paths in filenames
def fix_folder_structure(temp_dir, target_dir):
    """Move files with backslash names to proper folder structure."""
    for item in os.listdir(temp_dir):
        src_path = os.path.join(temp_dir, item)
        
        # Check if it's a file with backslash in name (Windows path issue)
        if os.path.isfile(src_path) and '\\' in item:
            # Convert backslash path to proper path
            proper_path = item.replace('\\', '/')
            dest_path = os.path.join(target_dir, proper_path)
            
            # Create parent directories
            os.makedirs(os.path.dirname(dest_path), exist_ok=True)
            
            # Move file
            shutil.copy2(src_path, dest_path)
            print(f"  Fixed: {item} → {proper_path}")
        
        elif os.path.isdir(src_path):
            # It's a proper directory, copy it directly
            dest_path = os.path.join(target_dir, item)
            if os.path.exists(dest_path):
                shutil.rmtree(dest_path)
            shutil.copytree(src_path, dest_path)
            print(f"  Copied dir: {item}/")
        
        else:
            # Regular file, copy directly
            dest_path = os.path.join(target_dir, item)
            shutil.copy2(src_path, dest_path)
            print(f"  Copied: {item}")

print("Fixing folder structure...")
fix_folder_structure(TEMP_DIR, PROJECT_ROOT)

# Also handle processed/ folder - move to data/processed/
if os.path.exists('/content/processed'):
    os.makedirs('/content/data/processed', exist_ok=True)
    for f in os.listdir('/content/processed'):
        src = os.path.join('/content/processed', f)
        dst = os.path.join('/content/data/processed', f)
        shutil.copy2(src, dst)
    shutil.rmtree('/content/processed')
    print("✓ Moved processed/ → data/processed/")

# Cleanup temp folder
shutil.rmtree(TEMP_DIR, ignore_errors=True)

# Set up Python path
import sys
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

print("\n✅ Project structure fixed!")
print("\nVerifying structure:")
print("\nSource modules:")
!ls -la src/
print("\nData modules:")
!ls -la src/data/
print("\nData files:")
!ls -la data/processed/

## 4️⃣ Load Dataset

In [ ]:
from src.data.tequad_loader import load_tequad_dataset, TeQuADDataset

# Load all splits
dataset = TeQuADDataset()
print(dataset)

# Show sample
print("\n📝 Sample from training set:")
sample = dataset.train[0]
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answers']['text'][0]}")
print(f"Context: {sample['context'][:200]}...")

## 5️⃣ XLM-R Training Configuration

In [ ]:
# XLM-R Training Configuration
CONFIG = {
    # Model
    "model_key": "xlmr",
    "model_name": "xlm-roberta-base",
    
    # Training params (optimized for Colab T4)
    "learning_rate": 3e-5,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 2,
    
    # Preprocessing
    "max_seq_length": 384,
    "doc_stride": 128,
    
    # Evaluation & Saving
    "eval_steps": 1000,
    "save_steps": 1000,
    
    # Output
    "output_dir": XLMR_OUTPUT,
}

print("⚙️ XLM-R Training Configuration:")
print("="*50)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

print(f"\n📊 Model Comparison:")
print(f"   MuRIL:     237M params (best for Telugu)")
print(f"   XLM-R:     270M params (cross-lingual)")
print(f"   IndicBERT: 33M params (lightweight)")

## 6️⃣ Load XLM-R Model

In [ ]:
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
import torch

print(f"Loading {CONFIG['model_name']}...")

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
model = AutoModelForQuestionAnswering.from_pretrained(CONFIG['model_name'])

# Move to GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Print model info
num_params = sum(p.numel() for p in model.parameters())
print(f"\n✓ Model loaded successfully!")
print(f"  Parameters: {num_params:,}")
print(f"  Device: {device}")

## 7️⃣ Preprocess Data

In [ ]:
from src.data.preprocessing import preprocess_for_training, preprocess_validation

print("Preprocessing training data for XLM-R...")
train_dataset = preprocess_for_training(
    dataset.train,
    tokenizer,
    max_length=CONFIG["max_seq_length"],
    doc_stride=CONFIG["doc_stride"],
    num_proc=1
)
print(f"✓ Training examples: {len(train_dataset)}")

print("\nPreprocessing validation data...")
eval_dataset = preprocess_for_training(
    dataset.validation,
    tokenizer,
    max_length=CONFIG["max_seq_length"],
    doc_stride=CONFIG["doc_stride"],
    num_proc=1
)
print(f"✓ Validation examples: {len(eval_dataset)}")

## 8️⃣ Setup Trainer

In [ ]:
from transformers import Trainer, TrainingArguments, default_data_collator

# Output directory for XLM-R
xlmr_output = os.path.join(CONFIG["output_dir"], "xlmr")
os.makedirs(xlmr_output, exist_ok=True)

# Calculate warmup steps (10% of total)
total_steps = (len(train_dataset) // (CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"])) * CONFIG["num_train_epochs"]
warmup_steps = int(0.1 * total_steps)

training_args = TrainingArguments(
    output_dir=xlmr_output,
    
    # Training
    learning_rate=CONFIG["learning_rate"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    
    # Optimization
    fp16=True,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    
    # Evaluation & Saving
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    # Logging
    logging_steps=100,
    logging_first_step=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)

print(f"\n✓ XLM-R Trainer ready!")
print(f"  Output: {xlmr_output}")
print(f"  Epochs: {CONFIG['num_train_epochs']}")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Effective batch size: {CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']}")
print(f"  Evaluation every {CONFIG['eval_steps']} steps")
print(f"\n📊 Watch for: eval_loss decreasing → good | eval_loss increasing → overfitting")

## 9️⃣ Train XLM-R 🚀

This will take approximately 2-3 hours on a T4 GPU.

In [ ]:
print("="*60)
print("🚀 Starting XLM-R Training")
print("="*60)

# Optionally resume from checkpoint
resume_checkpoint = None  # Set to checkpoint path to resume

train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

print("\n" + "="*60)
print("✅ XLM-R Training Complete!")
print("="*60)
print(f"\nMetrics:")
for key, value in train_result.metrics.items():
    print(f"  {key}: {value}")

## 🔟 Analyze Training Results

Look at the training logs above and find the checkpoint with the **lowest validation loss**.

In [ ]:
# List all checkpoints with their eval_loss
import json

trainer_state_path = os.path.join(xlmr_output, "trainer_state.json")
if os.path.exists(trainer_state_path):
    with open(trainer_state_path, 'r') as f:
        trainer_state = json.load(f)
    
    print("📊 Training History:")
    print("="*60)
    print(f"{'Step':<10} {'Train Loss':<15} {'Val Loss':<15}")
    print("-"*60)
    
    best_step = None
    best_val_loss = float('inf')
    
    for log in trainer_state.get('log_history', []):
        if 'eval_loss' in log:
            step = log.get('step', 0)
            train_loss = log.get('loss', 'N/A')
            val_loss = log.get('eval_loss', 'N/A')
            
            marker = ""
            if isinstance(val_loss, float) and val_loss < best_val_loss:
                best_val_loss = val_loss
                best_step = step
                marker = " ⭐ BEST"
            
            print(f"{step:<10} {train_loss:<15} {val_loss:<15}{marker}")
    
    print("\n" + "="*60)
    print(f"🏆 Best checkpoint: step {best_step} (val_loss: {best_val_loss:.4f})")
    print("="*60)
else:
    print("Training state not found. Check the checkpoints manually.")

# List available checkpoints
print("\n📁 Available checkpoints:")
!ls -la {xlmr_output}/checkpoint-*

## 1️⃣1️⃣ Save Best Model

**IMPORTANT:** Update `BEST_CHECKPOINT` below with the step number that had the lowest validation loss.

In [ ]:
import shutil

# ⚠️ UPDATE THIS with the best checkpoint step from the analysis above
BEST_CHECKPOINT = best_step if best_step else 9000  # Change this based on your training results

best_checkpoint_path = os.path.join(xlmr_output, f"checkpoint-{BEST_CHECKPOINT}")
xlmr_final = os.path.join(xlmr_output, "final")

# Copy best checkpoint as final model
if os.path.exists(best_checkpoint_path):
    shutil.copytree(best_checkpoint_path, xlmr_final, dirs_exist_ok=True)
    print(f"✓ Copied checkpoint-{BEST_CHECKPOINT} to: {xlmr_final}")
else:
    # Fallback: save current trainer state
    trainer.save_model(xlmr_final)
    tokenizer.save_pretrained(xlmr_final)
    print(f"✓ Saved trainer model to: {xlmr_final}")

# List saved files
print("\n📦 Saved files:")
for f in os.listdir(xlmr_final):
    size = os.path.getsize(os.path.join(xlmr_final, f))
    print(f"  {f}: {size/1024/1024:.1f} MB")

print(f"\n🏆 Best checkpoint: {BEST_CHECKPOINT}")

## 1️⃣2️⃣ Test XLM-R Model

In [ ]:
from transformers import pipeline

# Load the trained model
qa_pipeline = pipeline(
    "question-answering",
    model=xlmr_final,
    tokenizer=xlmr_final,
    device=0  # GPU
)

# Test cases
test_cases = [
    {
        "context": "హైదరాబాద్ తెలంగాణ రాష్ట్ర రాజధాని. ఇది దక్కన్ పీఠభూమిపై ఉంది. హైదరాబాద్ జనాభా దాదాపు 1 కోటి.",
        "question": "తెలంగాణ రాజధాని ఏది?"
    },
    {
        "context": "భారతదేశం దక్షిణ ఆసియాలో ఉన్న దేశం. భారతదేశ రాజధాని న్యూఢిల్లీ.",
        "question": "భారతదేశ రాజధాని ఏది?"
    },
    {
        "context": "విజయవాడ ఆంధ్రప్రదేశ్ లోని ముఖ్యమైన నగరం. ఇది కృష్ణా నది ఒడ్డున ఉంది.",
        "question": "విజయవాడ ఏ నది ఒడ్డున ఉంది?"
    }
]

print("🧪 XLM-R Test Results:")
print("="*70)

for i, tc in enumerate(test_cases, 1):
    result = qa_pipeline(question=tc['question'], context=tc['context'])
    print(f"\nTest {i}: {tc['question']}")
    print(f"  Answer: {result['answer']}")
    print(f"  Confidence: {result['score']:.2%}")

## 1️⃣3️⃣ Compare All Three Models (Optional)

If you have MuRIL and IndicBERT trained, run this cell to compare all three.

In [ ]:
from transformers import pipeline
import os

# Paths to all models (update these based on your training)
MODEL_PATHS = {
    "MuRIL": "/content/drive/MyDrive/telugu-qa-checkpoints/muril/checkpoint-9000",
    "IndicBERT": "/content/drive/MyDrive/telugu-qa-BERT-checkpoints/indicbert/final",
    "XLM-R": xlmr_final
}

# Load available models
pipelines = {}
for name, path in MODEL_PATHS.items():
    if os.path.exists(path):
        print(f"Loading {name}...")
        pipelines[name] = pipeline("question-answering", model=path, tokenizer=path, device=0)
    else:
        print(f"⚠️ {name} not found at {path}")

if len(pipelines) > 1:
    print("\n" + "="*80)
    print("📊 THREE-MODEL COMPARISON")
    print("="*80)
    
    test_context = "హైదరాబాద్ తెలంగాణ రాష్ట్ర రాజధాని. ఇది దక్కన్ పీఠభూమిపై ఉంది. హైదరాబాద్ జనాభా దాదాపు 1 కోటి."
    test_question = "తెలంగాణ రాజధాని ఏది?"
    
    print(f"\nQuestion: {test_question}")
    print(f"Context: {test_context[:80]}...")
    print("\n" + "-"*80)
    
    for name, pipe in pipelines.items():
        result = pipe(question=test_question, context=test_context)
        print(f"{name:12} → {result['answer']:20} (confidence: {result['score']:.2%})")
else:
    print("\nNeed at least 2 models for comparison.")

## 1️⃣4️⃣ Download XLM-R Model

In [ ]:
import shutil

xlmr_zip = "telugu-qa-xlmr-trained"
shutil.make_archive(xlmr_zip, 'zip', xlmr_final)

print(f"\n✓ XLM-R model zipped: {xlmr_zip}.zip")
print("\n📥 Download from Google Drive:")
print(f"   {XLMR_OUTPUT}/xlmr/final/")
print("\n📋 To use locally:")
print("   1. Download the zip")
print("   2. Extract to: models/checkpoints/xlmr/")
print("   3. Run evaluation: python scripts/evaluate_model.py --model xlmr")

# Optional: Direct download
# from google.colab import files
# files.download(f"{xlmr_zip}.zip")

---

## 📋 Training Summary

**Model Trained:** XLM-RoBERTa (xlm-roberta-base)

| Metric | Value |
|--------|-------|
| Parameters | 270M |
| Training Time | ~2-3 hours |
| Best Checkpoint | See cell above |
| Location | `telugu-qa-XLMR-checkpoints/xlmr/final` |

**Expected Performance Ranking:**
1. 🥇 MuRIL (~68% EM, ~84% F1) - Best for Telugu
2. 🥈 XLM-R (~50-60% EM, ~70-75% F1) - Cross-lingual
3. 🥉 IndicBERT (~10% EM, ~36% F1) - Lightweight

**Next Steps:**
1. Download XLM-R model zip
2. Extract to `models/checkpoints/xlmr/` locally
3. Run evaluation:
   ```bash
   python scripts/evaluate_model.py --model xlmr
   ```
4. Update Streamlit app to include XLM-R in model selector!